In [0]:
from medalhao_config import *



## silver.tb_info_filmes

In [0]:
from pyspark.sql import functions as F, Row
from datetime import datetime
from pyspark.sql.window import Window


dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

In [0]:
# pegando como referencia o uso de regex que foi falado na aula pratica, junto da parte que uso como base o notebook 

from medalhao_config import *
from pyspark.sql import functions as F
df_bronze_filmes=spark.table(table(bronze_schema,"tb_movies_info"))
                             
df_silver_filmes=(df_bronze_filmes
    .withColumnRenamed("id","id_filme")
    .withColumnRenamed("title","titulo")
    .withColumnRenamed("original_title","titulo_original")
    .withColumnRenamed("release_date","data_lancamento")
    .withColumnRenamed("runtime","duracao_minutos")
    .withColumnRenamed("original_language","idioma_original")
    .withColumnRenamed("status","status_filme")
    .withColumnRenamed("overview","sinopse")
    .withColumnRenamed("tagline","frase_divulgacao")
)

df_silver_filmes=df_silver_filmes.withColumn(
    "status_filme",
    F.when(F.upper(F.regexp_replace(F.trim("status_filme"),"[-_]"," "))=="RELEASED","Lançado")
     .when(F.upper(F.regexp_replace(F.trim("status_filme"),"[-_]"," "))=="POST PRODUCTION","Pós-Produção")
     .when(F.upper(F.regexp_replace(F.trim("status_filme"),"[-_]"," "))=="IN PRODUCTION","Em Produção")
     .when(F.upper(F.regexp_replace(F.trim("status_filme"),"[-_]"," "))=="PLANNED","Planejado")
     .when(F.upper(F.regexp_replace(F.trim("status_filme"),"[-_]"," "))=="RUMORED","Rumores")
     .when(F.upper(F.regexp_replace(F.trim("status_filme"),"[-_]"," "))=="CANCELED","Cancelado")
     .otherwise("Não Informado")
)

df_silver_filmes=df_silver_filmes.withColumn(
    "data_lancamento",
    F.coalesce(
        F.expr("try_to_date(data_lancamento,'yyyy-MM-dd')"),
        F.expr("try_to_date(data_lancamento,'MM-dd-yyyy')"),
        F.expr("try_to_date(data_lancamento,'dd-MM-yyyy')"),
        F.expr("try_to_date(data_lancamento,'MM/dd/yyyy')"),
        F.expr("try_to_date(data_lancamento,'dd/MM/yyyy')"),
        F.expr("try_to_date(data_lancamento,'yyyy/MM/dd')")
    )
).withColumn("ano_lancamento",F.year("data_lancamento"))


df_silver_filmes=df_silver_filmes.dropDuplicates(["id_filme"])

dq_check("silver.filmes","id_filme não nulo",df_silver_filmes,F.col("id_filme").isNotNull())
dq_check_unique("silver.filmes","id_filme único",df_silver_filmes,["id_filme"])

df_silver_filmes.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(table(silver_schema,"filmes"))


display(df_silver_filmes.limit(10))

[PASS] silver.filmes | id_filme não nulo | 0/97879 linhas falharam
[PASS] silver.filmes | id_filme único | 0 chaves duplicadas de 97879 linhas


id_filme,tconst,titulo,titulo_original,idioma_original,data_lancamento,duracao_minutos,status_filme,sinopse,frase_divulgacao,ingestion_datetime,ano_lancamento
260513,tt3606756,Incredibles 2,Incredibles 2,en,2018-06-14,118,Lançado,"Elastigirl springs into action to save the day, while Mr. Incredible faces his greatest challenge yet – taking care of the problems of his three children.",It's time to get back to being incredible.,2026-10-07T15:24:00.080Z,2018
278927,tt3040964,The Jungle Book,The Jungle Book,en,2016-04-07,106,Lançado,"A man-cub named Mowgli fostered by wolves. After a threat from the tiger Shere Khan, Mowgli is forced to flee the jungle, by which he embarks on a journey of self discovery with the help of the panther, Bagheera and the free-spirited bear, Baloo.",null,2026-10-07T15:24:00.080Z,2016
550988,tt6264654,Free Guy,Free Guy,en,2021-08-11,115,Lançado,"A bank teller discovers he is actually a background player in an open-world video game, and decides to become the hero of his own story. Now, in a world where there are no limits, he is determined to be the guy who saves his world his way before it's too late.",Life's too short to be a background character.,2026-10-07T15:24:00.080Z,2021
333371,tt1179933,10 Cloverfield Lane,10 Cloverfield Lane,en,2016-03-10,104,Lançado,"After getting in a car accident, a woman is held in a shelter with two men, who claim the outside world is affected by a widespread chemical attack.",Monsters come in many forms.,2026-10-07T15:24:00.080Z,2016
337401,tt4566758,Mulan,Mulan,en,2020-09-04,115,Lançado,"When the Emperor of China issues a decree that one man per family must serve in the Imperial Chinese Army to defend the country from Huns, Hua Mulan, the eldest daughter of an honored warrior, steps in to take the place of her ailing father. She is spirited, determined and quick on her feet. Disguised as a man by the name of Hua Jun, she is tested every step of the way and must harness her innermost strength and embrace her true potential.",Loyal. Brave. True.,2026-10-07T15:24:00.080Z,2020
500664,tt6499752,Upgrade,Upgrade,en,2018-05-31,100,Lançado,"A brutal mugging leaves Grey Trace paralyzed in the hospital and his beloved wife dead. A billionaire inventor soon offers Trace a cure — an artificial intelligence implant called STEM that will enhance his body. Now able to walk, Grey finds that he also has superhuman strength and agility — skills he uses to seek revenge against the thugs who destroyed his life.",Not man. Not machine. More.,2026-10-07T15:24:00.080Z,2018
481848,tt7504726,The Call of the Wild,The Call of the Wild,en,2020-02-19,100,Lançado,"Buck is a big-hearted dog whose blissful domestic life is turned upside down when he is suddenly uprooted from his California home and transplanted to the exotic wilds of the Yukon during the Gold Rush of the 1890s. As the newest rookie on a mail delivery dog sled team—and later its leader—Buck experiences the adventure of a lifetime, ultimately finding his true place in the world and becoming his own master.",Based on the legendary novel,2026-10-07T15:24:00.080Z,2020
635744,tt10767720,Jarhead: Law of Return,Jarhead: Law of Return,en,2019-10-01,103,Lançado,"Major Ronan Jackson, an accomplished fighter pilot for the Israel Defense Forces and son of a U.S. Senator, is shot down while flying through Syrian airspace. After miraculously surviving the crash, Jackson is taken captive by a group of Hezbollah militiamen. A squad of elite soldiers, led by Gunnery Sergeant Dave Torres, risk their own lives in the hopes of saving an ally they've never met.",null,2026-10-07T15:24:00.080Z,2019
353616,tt4765284,Pitch Perfect 3,Pitch Perfect 3,en,2017-12-20,93,Lançado,"After the highs of winning the world championships, the Bellas find themselves split apart and discovering there aren't job prospects for making music with your mouth. But when they get the chance to reunite for an overseas USO tour, this group of awesome nerds will come together to make some m

## silver.tb_financeiro_filmes (origem: tb_movies_financials)

In [0]:
# Silver: limpa valores financeiros, converte para decimal, aplica cotação e calcula lucro e margem. 

df_bronze_financeiro=spark.table(table(bronze_schema,"tb_movies_financials"))
df_bronze_cotacao=spark.table(table(bronze_schema,"tb_cotacao_dolar"))

df_silver_financeiro=(df_bronze_financeiro
    .withColumnRenamed("id","id_filme")
    .withColumnRenamed("budget","orcamento_usd")
    .withColumnRenamed("revenue","receita_usd")
)

df_silver_financeiro=df_silver_financeiro.select(
    "*",
    F.lit(None).cast("decimal(18,2)").alias("orcamento_brl"),
    F.lit(None).cast("decimal(18,2)").alias("receita_brl")
)

df_silver_financeiro=df_silver_financeiro.withColumn(
    "orcamento_usd",
    F.when(
        F.upper(F.trim(F.col("orcamento_usd").cast("string"))).isin("UNKNOWN","NÃO INFORMADO","NA","N/A","NULL",""),
        None
    ).otherwise(
        F.regexp_replace(
            F.regexp_replace(F.col("orcamento_usd").cast("string"),r"[A-Za-z$€£R\s]",""),
            r",",""
        ).cast("decimal(18,2)")
    )
).withColumn(
    "receita_usd",
    F.when(
        F.upper(F.trim(F.col("receita_usd").cast("string"))).isin("UNKNOWN","NÃO INFORMADO","NA","N/A","NULL",""),
        None
    ).otherwise(
        F.regexp_replace(
            F.regexp_replace(F.col("receita_usd").cast("string"),r"[A-Za-z$€£R\s]",""),
            r",",""
        ).cast("decimal(18,2)")
    )
)

df_silver_financeiro=df_silver_financeiro.withColumn(
    "orcamento_usd",
    F.when(F.col("orcamento_usd")>0,F.col("orcamento_usd"))
).withColumn(
    "receita_usd",
    F.when(F.col("receita_usd")>0,F.col("receita_usd"))
)

cotacao=df_bronze_cotacao.agg(
    F.last("cotacao_venda",True).alias("cotacao_venda")
).first()["cotacao_venda"]

df_silver_financeiro=df_silver_financeiro.withColumn(
    "orcamento_brl",
    (F.col("orcamento_usd")*F.lit(cotacao)).cast("decimal(18,2)")
).withColumn(
    "receita_brl",
    (F.col("receita_usd")*F.lit(cotacao)).cast("decimal(18,2)")
).withColumn(
    "lucro_usd",
    (F.col("receita_usd")-F.col("orcamento_usd")).cast("decimal(18,2)")
).withColumn(
    "lucro_brl",
    (F.col("receita_brl")-F.col("orcamento_brl")).cast("decimal(18,2)")
).withColumn(
    "margem_lucro_percentual",
    F.when(
        F.col("receita_usd").isNotNull() & (F.col("receita_usd")!=0),
        ((F.col("receita_usd")-F.col("orcamento_usd"))/F.col("receita_usd")*100).cast("decimal(10,2)")
    )
)
window_filmes=Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())

df_silver_financeiro=(df_silver_financeiro
    .withColumn("rn",F.row_number().over(window_filmes))
    .filter(F.col("rn")==1)
    .drop("rn")
)

dq_check("silver.tb_financeiro_filmes","id_filme não nulo",df_silver_financeiro,F.col("id_filme").isNotNull())
dq_check_unique("silver.tb_financeiro_filmes","id_filme único",df_silver_financeiro,["id_filme"])

df_silver_financeiro.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(table(silver_schema,"tb_financeiro_filmes"))

display(df_silver_financeiro.limit(10))

[PASS] silver.tb_financeiro_filmes | id_filme não nulo | 0/99006 linhas falharam
[PASS] silver.tb_financeiro_filmes | id_filme único | 0 chaves duplicadas de 99006 linhas


id_filme,orcamento_usd,receita_usd,ingestion_datetime,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
32471,null,null,2026-10-07T21:44:49.481Z,null,null,null,null,null
38492,null,null,2026-10-07T21:44:49.481Z,null,null,null,null,null
38700,90.00,426505244.00,2026-10-07T21:44:49.481Z,449.42,2129753935.91,426505154.00,2129753486.49,100.00
42018,null,null,2026-10-07T21:44:49.481Z,null,null,null,null,null
42330,null,null,2026-10-07T21:44:49.481Z,null,null,null,null,null
45033,337200.00,null,2026-10-07T21:44:49.481Z,1683808.20,null,null,null,null
50022,null,18850674.00,2026-10-07T21:44:49.481Z,null,94130840.62,null,null,null
58431,null,16457494.00,2026-10-07T21:44:49.481Z,null,82180496.29,null,null,null
66213,null,null,2026-10-07T21:44:49.481Z,null,null,null,null,null
66534,null,null,2026-10-07T21:44:49.481Z,null,null,null,null,null


tabela 3 

In [0]:
# Silver: limpa métricas, corrige escalas inválidas, trata column shift, remove duplicidades e valida os indicadores.



df_bronze_metricas=spark.table(table(bronze_schema,"tb_movies_metrics"))

df_silver_metricas=(df_bronze_metricas
    .withColumnRenamed("id","id_filme")
    .withColumnRenamed("popularity","popularidade")
    .withColumnRenamed("vote_average","nota_media_tmdb")
    .withColumnRenamed("vote_count","qtd_votos_tmdb")
    .withColumnRenamed("averageRating","nota_media_imdb")
    .withColumnRenamed("numVotes","qtd_votos_imdb")
)

window_filmes=Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())

df_silver_metricas=(df_silver_metricas
    .withColumn("rn",F.row_number().over(window_filmes))
    .filter(F.col("rn")==1)
    .drop("rn")
)

df_silver_metricas=df_silver_metricas.withColumn(
    "popularidade",
    F.expr("""
        try_cast(
            replace(
                regexp_replace(trim(cast(popularidade as string)), '[^0-9,.-]', ''),
                ',',
                '.'
            ) as double
        )
    """)
).withColumn(
    "nota_media_tmdb",
    F.expr("try_cast(trim(cast(nota_media_tmdb as string)) as double)")
).withColumn(
    "qtd_votos_tmdb",
    F.expr("try_cast(trim(cast(qtd_votos_tmdb as string)) as long)")
).withColumn(
    "nota_media_imdb",
    F.expr("try_cast(trim(cast(nota_media_imdb as string)) as double)")
).withColumn(
    "qtd_votos_imdb",
    F.expr("try_cast(trim(cast(qtd_votos_imdb as string)) as long)")
)

df_silver_metricas=df_silver_metricas.withColumn(
    "popularidade",
    F.when(F.col("popularidade")>=0,F.col("popularidade"))
).withColumn(
    "qtd_votos_tmdb",
    F.when(F.col("qtd_votos_tmdb")>=0,F.col("qtd_votos_tmdb"))
).withColumn(
    "qtd_votos_imdb",
    F.when(F.col("qtd_votos_imdb")>=0,F.col("qtd_votos_imdb"))
).withColumn(
    "nota_media_tmdb",
    F.when(F.col("nota_media_tmdb").between(0,10),F.col("nota_media_tmdb"))
).withColumn(
    "nota_media_imdb",
    F.when(F.col("nota_media_imdb").between(0,10),F.col("nota_media_imdb"))
)

dq_check("silver.tb_metricas_engajamento","id_filme não nulo",df_silver_metricas,F.col("id_filme").isNotNull())
dq_check_unique("silver.tb_metricas_engajamento","id_filme único",df_silver_metricas,["id_filme"])

df_silver_metricas.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(table(silver_schema,"tb_metricas_engajamento"))

display(df_silver_metricas.limit(10))

[PASS] silver.tb_metricas_engajamento | id_filme não nulo | 0/99013 linhas falharam
[PASS] silver.tb_metricas_engajamento | id_filme único | 0 chaves duplicadas de 99013 linhas


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb,ingestion_datetime
42018,3.403,6.63,27,6.8,1750,2026-10-07T21:44:52.865Z
42330,1.561,10.0,1,6.4,34,2026-10-07T21:44:52.865Z
66534,1.837,null,9,5.8,208,2026-10-07T21:44:52.865Z
91639,3.508,5.7,11,5.3,562,2026-10-07T21:44:52.865Z
146834,6.813,3.0,9,2.2,384,2026-10-07T21:44:52.865Z
171782,0.607,7.5,2,7.1,107,2026-10-07T21:44:52.865Z
184710,1.991,4.4,7,2.4,664,2026-10-07T21:44:52.865Z
203835,14.644,5.3,null,4.9,16953,2026-10-07T21:44:52.865Z
224337,0.6,6.0,1,7.1,13,2026-10-07T21:44:52.865Z
232253,1.462,null,2,8.5,38,2026-10-07T21:44:52.865Z


tabela 4 

In [0]:
# Silver: renomeia avaliações, trata notas inválidas, preenche comentários vazios e remove avaliações duplicadas.



df_bronze_avaliacoes=spark.table(table(bronze_schema,"tb_movies_reviews"))

df_silver_avaliacoes=(df_bronze_avaliacoes
    .withColumnRenamed("id","id_filme")
    .withColumnRenamed("nome","nome_usuario")
    .withColumnRenamed("nota","nota_usuario")
    .withColumnRenamed("comentario","comentario_usuario")
)

df_silver_avaliacoes=df_silver_avaliacoes.withColumn(
    "nota_usuario",
    F.expr("try_cast(trim(cast(nota_usuario as string)) as double)")
).withColumn(
    "nota_usuario",
    F.when(F.col("nota_usuario").between(0,10),F.col("nota_usuario"))
).withColumn(
    "comentario_usuario",
    F.when(
        F.col("comentario_usuario").isNull() |
        (F.trim(F.col("comentario_usuario"))==""),
        F.lit("Sem comentário")
    ).otherwise(F.trim(F.col("comentario_usuario")))
)

df_silver_avaliacoes=df_silver_avaliacoes.dropDuplicates([
    "id_filme",
    "nome_usuario",
    "nota_usuario",
    "comentario_usuario"
])

dq_check("silver.tb_avaliacoes_usuarios","id_filme não nulo",df_silver_avaliacoes,F.col("id_filme").isNotNull())
dq_check_unique(
    "silver.tb_avaliacoes_usuarios",
    "avaliação única",
    df_silver_avaliacoes,
    ["id_filme","nome_usuario","nota_usuario","comentario_usuario"]
)

df_silver_avaliacoes.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(
    table(silver_schema,"tb_avaliacoes_usuarios")
)

display(df_silver_avaliacoes.limit(10))

[PASS] silver.tb_avaliacoes_usuarios | id_filme não nulo | 0/32412 linhas falharam
[PASS] silver.tb_avaliacoes_usuarios | avaliação única | 0 chaves duplicadas de 32412 linhas


id_filme,nome_usuario,nota_usuario,comentario_usuario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,Sem comentário,2026-10-07T17:03:11.296Z
1152100,Isabela Pinto 509,7.6,Legal! Uma boa opção para o fim de semana.,2026-10-07T17:03:11.296Z
846452,Sérgio Freitas,6.1,Assisti até o final mas não me marcou.,2026-10-07T17:03:11.296Z
410937,Carolina Lopes 276,2.1,Horrível! Perda de tempo.,2026-10-07T17:03:11.296Z
1155082,Carolina Almeida 653,3.2,"Fraco, não recomendo.",2026-10-07T17:03:11.296Z
923930,Carlos Monteiro 465,9.0,Excepcional! História envolvente e atuações impecáveis.,2026-10-07T17:03:11.296Z
661409,Matheus Gomes 340,9.8,Perfeito! Um dos melhores filmes que já vi.,2026-10-07T17:03:11.296Z
447163,Natália Freitas 402,9.1,"Obra-prima do cinema, simplesmente espetacular.",2026-10-07T17:03:11.296Z
464175,Mônica Cavalcanti,7.7,Filme interessante com boas atuações.,2026-10-07T17:03:11.296Z
1020969,Carolina Almeida 690,3.2,"Ruim, não vale o tempo investido.",2026-10-07T17:03:11.296Z


tabela five 

In [0]:
# Silver: normaliza separadores, explode os gêneros e remove resíduos inválidos da origem.



df_bronze_generos=spark.table(table(bronze_schema,"tb_credits_and_tags"))

df_silver_generos=(df_bronze_generos
    .select("id","genres")
    .withColumnRenamed("id","id_filme")
)

df_silver_generos=df_silver_generos.withColumn(
    "genero",
    F.explode(
        F.split(
            F.regexp_replace(F.col("genres"),";",","),
            ","
        )
    )
)

df_silver_generos=df_silver_generos.withColumn(
    "genero",
    F.trim(
        F.regexp_replace(
            F.col("genero"),
            r"^[\[\]{}'\"\s]+|[\[\]{}'\"\s]+$",
            ""
        )
    )
)

df_silver_generos=df_silver_generos.filter(
    (F.col("genero").isNotNull()) &
    (F.col("genero")!="") &
    ~F.col("genero").rlike(r"^[0-9]+([.,][0-9]+)?$")
)

generos_validos=[
    "Action","Adventure","Animation","Comedy","Crime","Documentary",
    "Drama","Family","Fantasy","History","Horror","Music","Mystery",
    "Romance","Science Fiction","TV Movie","Thriller","War","Western"
]

df_silver_generos=df_silver_generos.filter(
    F.col("genero").isin(generos_validos)
)

df_silver_generos=df_silver_generos.dropDuplicates(["id_filme","genero"])

dq_check("silver.tb_generos","id_filme não nulo",df_silver_generos,F.col("id_filme").isNotNull())
dq_check_unique("silver.tb_generos","filme e gênero únicos",df_silver_generos,["id_filme","genero"])

df_silver_generos.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(
    table(silver_schema,"tb_generos")
)

display(df_silver_generos.limit(20))

[PASS] silver.tb_generos | id_filme não nulo | 0/132971 linhas falharam
[PASS] silver.tb_generos | filme e gênero únicos | 0 chaves duplicadas de 132971 linhas


id_filme,genres,genero
259316,"Fantasy, Adventure, Family",Adventure
354912,"Family, Animation, Fantasy, Music, Comedy, Adventure",Family
420817,"Adventure, Fantasy, Romance, Family",Romance
420817,"Adventure, Fantasy, Romance, Family",Family
438631,"Science Fiction, Adventure",Science Fiction
283366,"Fantasy, Adventure",Adventure
580489,"Science Fiction, Action, Adventure",Adventure
550988,"Comedy, Adventure, Science Fiction",Comedy
339964,"Adventure, Science Fiction, Action",Action
454983,"Romance, Comedy",Comedy


tabela six 

In [0]:
# Silver: unifica atores, diretores, roteiristas e produtoras em uma única dimensão categorizada.


df_bronze_pessoas=spark.table(table(bronze_schema,"tb_credits_and_tags"))

df_silver_pessoas=(df_bronze_pessoas
    .select("id","cast","directors","writers","production_companies")
    .withColumnRenamed("id","id_filme")
)

df_silver_pessoas=df_silver_pessoas.selectExpr(
    "id_filme",
    """
    stack(
        4,
        'Ator', cast,
        'Diretor', directors,
        'Roteirista', writers,
        'Produtora', production_companies
    ) as (tipo_entidade, entidades)
    """
)

df_silver_pessoas=df_silver_pessoas.withColumn(
    "entidade",
    F.explode(
        F.split(
            F.regexp_replace(F.col("entidades"),";",","),
            ","
        )
    )
).drop("entidades")

df_silver_pessoas=df_silver_pessoas.withColumn(
    "entidade",
    F.trim(
        F.regexp_replace(
            F.col("entidade"),
            r"^[\[\]{}'\"\s]+|[\[\]{}'\"\s]+$",
            ""
        )
    )
)

df_silver_pessoas=df_silver_pessoas.withColumn(
    "entidade",
    F.initcap(F.lower(F.col("entidade")))
)

df_silver_pessoas=df_silver_pessoas.filter(
    (F.col("id_filme").isNotNull()) &
    (F.col("tipo_entidade").isNotNull()) &
    (F.col("entidade").isNotNull()) &
    (F.col("entidade")!="") &
    ~F.col("entidade").rlike(r"^[0-9]+([.,][0-9]+)?$")
)

df_silver_pessoas=df_silver_pessoas.dropDuplicates(
    ["id_filme","tipo_entidade","entidade"]
)

dq_check(
    "silver.tb_pessoas_empresas",
    "id_filme não nulo",
    df_silver_pessoas,
    F.col("id_filme").isNotNull()
)

dq_check(
    "silver.tb_pessoas_empresas",
    "entidade não nula",
    df_silver_pessoas,
    F.col("entidade").isNotNull() & (F.col("entidade")!="")
)

dq_check_unique(
    "silver.tb_pessoas_empresas",
    "filme, tipo e entidade únicos",
    df_silver_pessoas,
    ["id_filme","tipo_entidade","entidade"]
)

df_silver_pessoas.write.format("delta").mode("overwrite").option(
    "overwriteSchema","true"
).saveAsTable(table(silver_schema,"tb_pessoas_empresas"))

display(df_silver_pessoas.limit(20))

[PASS] silver.tb_pessoas_empresas | id_filme não nulo | 0/902534 linhas falharam
[PASS] silver.tb_pessoas_empresas | entidade não nula | 0/902534 linhas falharam
[PASS] silver.tb_pessoas_empresas | filme, tipo e entidade únicos | 0 chaves duplicadas de 902534 linhas


id_filme,tipo_entidade,entidade
447332,Ator,Noah Jupe
260513,Ator,Craig T. Nelson
399055,Ator,Richard Jenkins
332562,Ator,Andrew Dice Clay
351286,Ator,Toby Jones
337339,Ator,Michelle Rodriguez
293167,Ator,Toby Kebbell
359940,Ator,Lucas Hedges
405774,Ator,Sarah Paulson
353081,Ator,Vanessa Kirby


In [0]:
# Silver: cria série diária contínua e aplica Forward Fill da última cotação disponível. 


df_bronze_cotacao=spark.table(table(bronze_schema,"tb_cotacao_dolar"))

df_cotacao=(df_bronze_cotacao
    .withColumn("data_cotacao",F.to_date("data_hora_cotacao"))
    .select("data_cotacao","cotacao_compra","cotacao_venda")
    .groupBy("data_cotacao")
    .agg(
        F.last("cotacao_compra",True).alias("cotacao_compra"),
        F.last("cotacao_venda",True).alias("cotacao_venda")
    )
)

periodo=df_cotacao.select(
    F.min("data_cotacao").alias("data_inicio"),
    F.max("data_cotacao").alias("data_fim")
).first()

df_datas=spark.sql(
    f"""
    SELECT explode(
        sequence(
            to_date('{periodo["data_inicio"]}'),
            to_date('{periodo["data_fim"]}'),
            interval 1 day
        )
    ) AS data_cotacao
    """
)

df_silver_cotacao=df_datas.join(
    df_cotacao,
    on="data_cotacao",
    how="left"
)

window_ffill=Window.orderBy("data_cotacao").rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)

df_silver_cotacao=(df_silver_cotacao
    .withColumn(
        "cotacao_compra",
        F.last("cotacao_compra",True).over(window_ffill)
    )
    .withColumn(
        "cotacao_venda",
        F.last("cotacao_venda",True).over(window_ffill)
    )
)

dq_check(
    "silver.tb_cotacao_dolar",
    "data_cotacao não nula",
    df_silver_cotacao,
    F.col("data_cotacao").isNotNull()
)

dq_check(
    "silver.tb_cotacao_dolar",
    "cotacao_venda não nula",
    df_silver_cotacao,
    F.col("cotacao_venda").isNotNull()
)

dq_check(
    "silver.tb_cotacao_dolar",
    "cotacao_compra não nula",
    df_silver_cotacao,
    F.col("cotacao_compra").isNotNull()
)

dq_check_unique(
    "silver.tb_cotacao_dolar",
    "data_cotacao única",
    df_silver_cotacao,
    ["data_cotacao"]
)

df_silver_cotacao.write.format("delta").mode("overwrite").option(
    "overwriteSchema","true"
).saveAsTable(table(silver_schema,"tb_cotacao_dolar"))

display(df_silver_cotacao.orderBy("data_cotacao"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] silver.tb_cotacao_dolar | data_cotacao não nula | 0/37 linhas falharam
[PASS] silver.tb_cotacao_dolar | cotacao_venda não nula | 0/37 linhas falharam
[PASS] silver.tb_cotacao_dolar | cotacao_compra não nula | 0/37 linhas falharam
[PASS] silver.tb_cotacao_dolar | data_cotacao única | 0 chaves duplicadas de 37 linhas


data_cotacao,cotacao_compra,cotacao_venda
2026-09-01,5.1564,5.157
2026-09-02,5.1267,5.1273
2026-09-03,5.0956,5.0962
2026-09-04,5.1247,5.1253
2026-09-05,5.1247,5.1253
2026-09-06,5.1247,5.1253
2026-09-07,5.1247,5.1253
2026-09-08,5.085,5.0856
2026-09-09,5.0973,5.0979
2026-09-10,5.1143,5.1149
